# Parametric PINN Validation Notebook

This notebook loads a trained parametric PINN model and validates it on all held-out cases not used during training. It computes RMSE for each case, generates comparison plots, and saves results.

## Setup

In [1]:
import sys
from pathlib import Path
from dataclasses import dataclass
import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Could not find project root containing /src")
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import set_seed, get_device
from src.data import load_manifest_rows, load_case_manifest_row
from src.pinn import MLP, LossWeights

set_seed(42)
device = get_device()
print(f"Project root: {ROOT}")
print(f"Device: {device}")

PART = "2026_03_16_1550"
# Specify the path to the trained model checkpoint
MODEL_CHECKPOINT = ROOT / "outputs" / "parametric" / PART   
MODEL_CHECKPOINT_PATH = MODEL_CHECKPOINT / "hard_left_bc" / "2026_03_16_1550_hard_left_bc_best.pt"   

if not MODEL_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Model checkpoint not found at {MODEL_CHECKPOINT_PATH}. Please update MODEL_CHECKPOINT_PATH.")

MU_TIME_SAMPLES = 15

# Create validation output directory
VAL_RUN_ID = datetime.now(timezone.utc).strftime("%Y_%m_%d_%H%M")
VAL_OUTDIR = MODEL_CHECKPOINT / "validation" / VAL_RUN_ID
VAL_OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Validation RUN_ID: {VAL_RUN_ID}")
print(f"Validation OUTDIR: {VAL_OUTDIR}")

Project root: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN
Device: cpu
Validation RUN_ID: 2026_03_16_1705
Validation OUTDIR: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\validation\2026_03_16_1705


## Load Trained Model

In [2]:
# Load the checkpoint
checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=device)
print(f"Loaded checkpoint from: {MODEL_CHECKPOINT_PATH}")

state_dict = checkpoint["state_dict"]
mu_dim = checkpoint.get("mu_dim", 30)

# Auto-detect whether this is the hard-left-BC wrapper or a plain MLP.
is_hard_left_bc = any(k.startswith("base.net.") for k in state_dict.keys())

if is_hard_left_bc:
    from src.pinn import LeftBoundaryAnsatzMLP
    model = LeftBoundaryAnsatzMLP(
        in_dim=2 + mu_dim,
        hidden=64,
        layers=3,
    ).to(device)
    print("Detected hard-left-BC checkpoint (theta = xi * N).")
else:
    from src.pinn import MLP
    model = MLP(
        in_dim=2 + mu_dim,
        hidden=64,
        layers=3,
    ).to(device)
    print("Detected standard soft-BC checkpoint.")

model.load_state_dict(state_dict)
model.eval()

print("Model loaded and set to evaluation mode.")
print(f"Model input dimension: {2 + mu_dim}")


Loaded checkpoint from: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\hard_left_bc\2026_03_16_1550_hard_left_bc_best.pt
Detected hard-left-BC checkpoint (theta = xi * N).
Model loaded and set to evaluation mode.
Model input dimension: 32


## Load Validation Cases

In [3]:
manifest_testing = ROOT / "data" / "manifest_testing.csv"
rows_testing = load_manifest_rows(manifest_testing)

# All cases in manifest_testing are new and unused for training
val_rows = rows_testing

# Load training manifest to compute mu_stats (normalization parameters from training)
manifest_train = ROOT / "data" / "manifest.csv"
rows_train = load_manifest_rows(manifest_train)

# Replicate the deterministic split from training to get train_rows for mu_stats
rows_sorted_train = sorted(rows_train, key=lambda r: r["case_id"])
n_val_cases_train = max(1, int(round(0.2 * len(rows_sorted_train))))
train_rows = rows_sorted_train[:-n_val_cases_train]  # First 80% are train cases

print("Loaded training cases for mu_stats:", [r["case_id"] for r in train_rows])
print("Validation cases (all new):", [r["case_id"] for r in val_rows])

# Load training cases to compute mu_stats (needed for normalization)
def compute_mu_stats(cases):
    eps = 1e-8
    all_mu = np.array([c["mu_raw"] for c in cases])
    mu_min = all_mu.min(axis=0)
    mu_max = all_mu.max(axis=0)
    return {
        "mu_min": mu_min,
        "mu_max": mu_max,
        "eps": eps,
    }

def normalise_mu(mu_raw, mu_stats):
    eps = mu_stats["eps"]
    mu_min = mu_stats["mu_min"]
    mu_max = mu_stats["mu_max"]
    return 2.0 * (mu_raw - mu_min) / (mu_max - mu_min + eps) - 1.0

def _load_case_with_mu(row: dict) -> dict:
    c = load_case_manifest_row(row, root=ROOT)
    
    # Match the current training notebook: right-boundary temperature history only.
    tau = c["nondim"]["tau"]
    theta = c["nondim"]["theta"]
    
    tau_max = tau.max()
    tau_samples = np.linspace(0, tau_max, MU_TIME_SAMPLES)
    theta_series_1 = np.interp(tau_samples, tau, theta[-1, :]).astype(np.float32)
    dtheta_dtau_1 = np.gradient(theta_series_1, tau_samples).astype(np.float32)
    
    mu_raw = np.concatenate([theta_series_1, dtheta_dtau_1]).astype(np.float32)
    
    c["mu_raw"] = mu_raw
    c["mu"] = mu_raw  # Will be normalized later
    return c

train_cases = [_load_case_with_mu(r) for r in train_rows]
val_cases = [_load_case_with_mu(r) for r in val_rows]

mu_stats = compute_mu_stats(train_cases)
for c in (train_cases + val_cases):
    c["mu"] = normalise_mu(c["mu_raw"], mu_stats)

print(f"Loaded {len(train_cases)} train cases and {len(val_cases)} validation cases.")
print(f"mu_dim: {len(val_cases[0]['mu'])}")

Loaded training cases for mu_stats: ['const_10000', 'const_15000', 'const_5000', 'offset_sine_1', 'sine_A10000_T100', 'sine_A10000_T50']
Validation cases (all new): ['const_6000', 'const_12000', 'const_14000', 'const_20000', 'sine_A5000_T50', 'sine_A7500_T75', 'sine_A7500_T125', 'sine_A7500_T25', 'sine_A12500_T75', 'sine_A12500_T125', 'sine_A12500_T25', 'const_10000_Tleft_280', 'const_10000_Tleft_300', 'const_10000_Tleft_310']
Loaded 6 train cases and 14 validation cases.
mu_dim: 30


## Compute Validation Metrics

In [4]:
# Compute RMSE for each validation case
validation_results = []

for i, test_case in enumerate(val_cases):
    case_id = test_case["case_id"]
    print(f"Computing metrics for case {i+1}/{len(val_cases)}: {case_id}")
    
    # Extract data
    xi = test_case["nondim"]["xi"]
    tau = test_case["nondim"]["tau"]
    theta_true = test_case["nondim"]["theta"]
    mu_case = test_case["mu"]
    
    # Predict
    xi_grid, tau_grid = np.meshgrid(xi, tau, indexing="ij")
    xi_flat = xi_grid.flatten()
    tau_flat = tau_grid.flatten()
    mu_flat = np.tile(mu_case, (xi_flat.size, 1))
    
    X_pred = np.column_stack([xi_flat, tau_flat, mu_flat]).astype(np.float32)
    X_pred_tensor = torch.tensor(X_pred, device=device)
    
    with torch.no_grad():
        theta_pred_flat = model(X_pred_tensor).cpu().numpy().flatten()
    
    theta_pred = theta_pred_flat.reshape(xi_grid.shape)
    
    # Compute RMSE
    mse = np.mean((theta_pred - theta_true) ** 2)
    rmse = np.sqrt(mse)
    
    validation_results.append({
        "case_id": case_id,
        "rmse": rmse,
        "mse": mse
    })
    
    print(f"  RMSE: {rmse:.4e}")

# Summary
rmse_values = [r["rmse"] for r in validation_results]
mean_rmse = np.mean(rmse_values)
std_rmse = np.std(rmse_values)
min_rmse = np.min(rmse_values)
max_rmse = np.max(rmse_values)

print("Validation Summary:")
print(f"  Number of cases: {len(validation_results)}")
print(f"  Mean RMSE: {mean_rmse:.4e}")
print(f"  Std RMSE: {std_rmse:.4e}")
print(f"  Min RMSE: {min_rmse:.4e}")
print(f"  Max RMSE: {max_rmse:.4e}")

Computing metrics for case 1/14: const_6000
  RMSE: 2.9002e-03
Computing metrics for case 2/14: const_12000
  RMSE: 2.9021e-03
Computing metrics for case 3/14: const_14000
  RMSE: 2.8994e-03
Computing metrics for case 4/14: const_20000
  RMSE: 2.9037e-03
Computing metrics for case 5/14: sine_A5000_T50
  RMSE: 1.7724e-03
Computing metrics for case 6/14: sine_A7500_T75
  RMSE: 2.2944e-01
Computing metrics for case 7/14: sine_A7500_T125
  RMSE: 5.6965e-01
Computing metrics for case 8/14: sine_A7500_T25
  RMSE: 5.7908e-01
Computing metrics for case 9/14: sine_A12500_T75
  RMSE: 2.2944e-01
Computing metrics for case 10/14: sine_A12500_T125
  RMSE: 5.6964e-01
Computing metrics for case 11/14: sine_A12500_T25
  RMSE: 5.7909e-01
Computing metrics for case 12/14: const_10000_Tleft_280
  RMSE: 2.9015e-03
Computing metrics for case 13/14: const_10000_Tleft_300
  RMSE: 2.9015e-03
Computing metrics for case 14/14: const_10000_Tleft_310
  RMSE: 2.9015e-03
Validation Summary:
  Number of cases: 14
  

## Plot Validation Results

In [5]:
# Generate plots for each validation case (saved to VAL_OUTDIR)
for i, test_case in enumerate(val_cases):
    case_id = test_case["case_id"]
    print(f"Generating plots for case {i+1}/{len(val_cases)}: {case_id}")
    
    # Extract data
    xi = test_case["nondim"]["xi"]
    tau = test_case["nondim"]["tau"]
    theta_true = test_case["nondim"]["theta"]
    mu_case = test_case["mu"]
    
    # Physical scales
    T_left = float(test_case["physical"]["T_ref"])
    dT = float(test_case["physical"]["dT_ref"])
    
    # Predict (already done above, but recompute for clarity)
    xi_grid, tau_grid = np.meshgrid(xi, tau, indexing="ij")
    xi_flat = xi_grid.flatten()
    tau_flat = tau_grid.flatten()
    mu_flat = np.tile(mu_case, (xi_flat.size, 1))
    
    X_pred = np.column_stack([xi_flat, tau_flat, mu_flat]).astype(np.float32)
    X_pred_tensor = torch.tensor(X_pred, device=device)
    
    with torch.no_grad():
        theta_pred_flat = model(X_pred_tensor).cpu().numpy().flatten()
    
    theta_pred = theta_pred_flat.reshape(xi_grid.shape)
    
    # Convert to physical temperature
    T_true_phys = T_left + dT * theta_true
    T_pred_phys = T_left + dT * theta_pred
    
    # Plot 1: Boundary profiles
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    
    axes[0].plot(tau, theta_true[0, :], 'k-', label='True', linewidth=2)
    axes[0].plot(tau, theta_pred[0, :], 'r--', label='Predicted', linewidth=2)
    axes[0].set_title(f'Boundary Temperature at xi=0 (Case {case_id})')
    axes[0].set_xlabel('tau')
    axes[0].set_ylabel('theta')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(tau, theta_true[-1, :], 'k-', label='True', linewidth=2)
    axes[1].plot(tau, theta_pred[-1, :], 'r--', label='Predicted', linewidth=2)
    axes[1].set_title(f'Boundary Temperature at xi=1 (Case {case_id})')
    axes[1].set_xlabel('tau')
    axes[1].set_ylabel('theta')
    axes[1].legend()
    axes[1].grid(True)
    
    boundary_plot_path = VAL_OUTDIR / f"boundary_profiles_{case_id}.png"
    plt.savefig(boundary_plot_path, dpi=150, bbox_inches='tight')
    plt.close()  # Don't show in notebook to avoid clutter
    
    # Plot 2: Temperature field comparison
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    
    im0 = axes[0].imshow(T_true_phys, aspect='auto', origin='lower',
                      extent=[xi.min(), xi.max(), tau.min(), tau.max()])
    axes[0].set_title(f'Ground Truth T (Case {case_id})')
    axes[0].set_xlabel('xi')
    axes[0].set_ylabel('tau')
    plt.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(T_pred_phys, aspect='auto', origin='lower',
                      extent=[xi.min(), xi.max(), tau.min(), tau.max()])
    axes[1].set_title(f'Predicted T (Case {case_id})')
    axes[1].set_xlabel('xi')
    axes[1].set_ylabel('tau')
    plt.colorbar(im1, ax=axes[1])
    
    temp_plot_path = VAL_OUTDIR / f"temperature_comparison_{case_id}.png"
    plt.savefig(temp_plot_path, dpi=150, bbox_inches='tight')
    plt.close()

print(f"All plots saved to: {VAL_OUTDIR}")

Generating plots for case 1/14: const_6000
Generating plots for case 2/14: const_12000
Generating plots for case 3/14: const_14000
Generating plots for case 4/14: const_20000
Generating plots for case 5/14: sine_A5000_T50
Generating plots for case 6/14: sine_A7500_T75
Generating plots for case 7/14: sine_A7500_T125
Generating plots for case 8/14: sine_A7500_T25
Generating plots for case 9/14: sine_A12500_T75
Generating plots for case 10/14: sine_A12500_T125
Generating plots for case 11/14: sine_A12500_T25
Generating plots for case 12/14: const_10000_Tleft_280
Generating plots for case 13/14: const_10000_Tleft_300
Generating plots for case 14/14: const_10000_Tleft_310
All plots saved to: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\validation\2026_03_16_1705


## Save Metrics and Plots

In [6]:
# Save validation metrics to CSV
import pandas as pd

metrics_df = pd.DataFrame(validation_results)
metrics_csv_path = VAL_OUTDIR / "validation_metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)

# Save summary to text file
summary_path = VAL_OUTDIR / "validation_summary.txt"
with open(summary_path, 'w') as f:
    f.write("Validation Summary\n")
    f.write("==================\n")
    f.write(f"Model checkpoint: {MODEL_CHECKPOINT_PATH}\n")
    f.write(f"Number of validation cases: {len(validation_results)}\n")
    f.write(f"Mean RMSE: {mean_rmse:.6e}\n")
    f.write(f"Std RMSE: {std_rmse:.6e}\n")
    f.write(f"Min RMSE: {min_rmse:.6e}\n")
    f.write(f"Max RMSE: {max_rmse:.6e}\n")
    f.write("\nPer-case RMSE:\n")
    for r in validation_results:
        f.write(f"{r['case_id']}: {r['rmse']:.6e}\n")

print(f"Metrics saved to: {metrics_csv_path}")
print(f"Summary saved to: {summary_path}")
print(f"All outputs saved in: {VAL_OUTDIR}")

Metrics saved to: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\validation\2026_03_16_1705\validation_metrics.csv
Summary saved to: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\validation\2026_03_16_1705\validation_summary.txt
All outputs saved in: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\validation\2026_03_16_1705


## Sensitivity Test to Mu

In [7]:
# Test if the model is sensitive to mu by comparing predictions with different mu vectors
# Select one unseen sine case
sine_cases = [c for c in val_cases if c['case_id'].startswith('sine')]
if not sine_cases:
    print("No sine cases in validation set, skipping sensitivity test.")
else:
    test_case = sine_cases[1]  # Pick the first sine case
    case_id = test_case['case_id']
    print(f"Testing sensitivity on case: {case_id}")
    
    # Function to compute RMSE for a given mu
    def compute_rmse(test_case, mu_vector, model, device):
        xi = test_case["nondim"]["xi"]
        tau = test_case["nondim"]["tau"]
        theta_true = test_case["nondim"]["theta"]
        
        xi_grid, tau_grid = np.meshgrid(xi, tau, indexing="ij")
        xi_flat = xi_grid.flatten()
        tau_flat = tau_grid.flatten()
        mu_flat = np.tile(mu_vector, (xi_flat.size, 1))
        
        X_pred = np.column_stack([xi_flat, tau_flat, mu_flat]).astype(np.float32)
        X_pred_tensor = torch.tensor(X_pred, device=device)
        
        with torch.no_grad():
            theta_pred_flat = model(X_pred_tensor).cpu().numpy().flatten()
        
        theta_pred = theta_pred_flat.reshape(xi_grid.shape)
        mse = np.mean((theta_pred - theta_true) ** 2)
        return np.sqrt(mse)
    
    # Test A: Correct mu
    mu_a = test_case['mu']
    rmse_a = compute_rmse(test_case, mu_a, model, device)
    print(f"Test A (correct mu): RMSE = {rmse_a:.4e}")
    
    # Test B: Replace mu with a different sine case (e.g., sine_A5000_T50)
    other_case_id = 'sine_A5000_T50'
    other_case = next((c for c in train_cases + val_cases if c['case_id'] == other_case_id), None)
    if other_case:
        mu_b = other_case['mu']
        rmse_b = compute_rmse(test_case, mu_b, model, device)
        print(f"Test B (mu from {other_case_id}): RMSE = {rmse_b:.4e}")
    else:
        print(f"Test B: Case {other_case_id} not found, skipping.")
        rmse_b = None
    
    # Test C: Zero out mu (leave only query coordinates)
    mu_c = np.zeros_like(mu_a)
    rmse_c = compute_rmse(test_case, mu_c, model, device)
    print(f"Test C (zero mu): RMSE = {rmse_c:.4e}")
    
    # Diagnosis
    if rmse_b is not None and abs(rmse_a - rmse_b) < 1e-6 and abs(rmse_a - rmse_c) < 1e-6:
        print("Diagnosis: Model appears insensitive to mu (predictions A, B, C are nearly identical).")
    else:
        print("Diagnosis: Model shows sensitivity to mu (predictions differ).")

Testing sensitivity on case: sine_A7500_T75
Test A (correct mu): RMSE = 2.2944e-01
Test B (mu from sine_A5000_T50): RMSE = 3.5527e-01
Test C (zero mu): RMSE = 4.3818e-01
Diagnosis: Model shows sensitivity to mu (predictions differ).


## Inspect Mu for Specific Cases

In [8]:
# Inspect mu vectors for specific cases to check for potential issues
cases_to_inspect = ['sine_A7500_T75', 'sine_A12500_T125', 'sine_A5000_T50']  # Examples: one from training, one unseen, one reference

for cid in cases_to_inspect:
    case = next((c for c in train_cases + val_cases if c['case_id'] == cid), None)
    if case:
        mu_raw = case['mu_raw']
        print(f"\nCase: {cid}")
        print(f"  Shape: {mu_raw.shape}")
        print(f"  Min: {mu_raw.min():.4f}, Max: {mu_raw.max():.4f}")
        print(f"  Mean: {mu_raw.mean():.4f}, Std: {mu_raw.std():.4f}")
        print(f"  First 10 values: {mu_raw[:10]}")
        print(f"  Last 10 values: {mu_raw[-10:]}")
        
        # Plot the time-series components
        tau_max = case['nondim']['tau'].max()
        tau_samples = np.linspace(0, tau_max, MU_TIME_SAMPLES)
        
        theta_1 = mu_raw[:MU_TIME_SAMPLES]
        dtheta_1 = mu_raw[MU_TIME_SAMPLES:]
        
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].plot(tau_samples, theta_1, 'r-')
        axes[0].set_title(f'Theta at xi=1 ({cid})')
        axes[0].set_xlabel('tau')
        axes[0].set_ylabel('theta')
        axes[0].grid(True)
        
        axes[1].plot(tau_samples, dtheta_1, 'm-')
        axes[1].set_title(f'dTheta/dTau at xi=1 ({cid})')
        axes[1].set_xlabel('tau')
        axes[1].set_ylabel('dtheta/dtau')
        axes[1].grid(True)
        
        plt.tight_layout()
        plot_path = VAL_OUTDIR / f"mu_components_{cid}.png"
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Plot saved: {plot_path}")
    else:
        print(f"Case {cid} not found in loaded cases.")


Case: sine_A7500_T75
  Shape: (30,)
  Min: -2.0771, Max: 2.8885
  Mean: -0.1086, Std: 1.4231
  First 10 values: [ 0.         -0.09626418 -0.2640619  -0.45987508 -0.6544936  -0.82227075
 -0.94179857 -0.99732524 -0.9798309  -0.8876568 ]
  Last 10 values: [-1.5284503  -0.9312824  -0.20233048  0.58343136  1.3481325   2.0175009
  2.526991    2.8273056   2.8885431   2.856995  ]
  Plot saved: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_16_1550\validation\2026_03_16_1705\mu_components_sine_A7500_T75.png

Case: sine_A12500_T125
  Shape: (30,)
  Min: -1.3419, Max: 0.6475
  Mean: -0.6269, Std: 0.5206
  First 10 values: [ 0.         -0.04965065 -0.13892409 -0.25023594 -0.3739256  -0.5017193
 -0.62616396 -0.7406135  -0.83929217 -0.91736674]
  Last 10 values: [-1.3418972  -1.270907   -1.1338329  -0.9403196  -0.70072925 -0.426042
 -0.12767987  0.18270394  0.4934765   0.6475177 ]
  Plot saved: c:\Users\wscm13\OneDrive - Loughborough